## **Generating Points**

### **Complete United States**

In [52]:
import pandas
import numpy
import random

lat_min = 34.551731
lat_max = 47.519609
lon_min = -120.166695
lon_max = -88.443061

coords = set()

while len(coords) < 100000:
    lat = numpy.random.uniform(lat_min, lat_max)
    lon = numpy.random.uniform(lon_min, lon_max)
    coord = (lon,lat)
    coords.add(coord)

coords = numpy.array(list(coords))

numpy.savetxt("Generated Coordinates (US).txt", coords, delimiter = ",")

globals().clear()

### **California**

### **Digital Elevation Model With Download**

In [ ]:
# %% [markdown]
# ### **United State Geological Survey Digital Elevation Model Query**

# %% [markdown]
# The following program uses Python to access USGS Digital Elevation Models (DEMs) to query elevation data for flight data.

# %% [markdown]
# Downloading Required Dependancies

# %%
!pip3 install tkinter
!pip3 install os
!pip3 install math
!pip3 install requests
!pip3 install time
!pip3 install pandas
!pip3 install tqdm
!pip3 install rasterio
!pip3 install pyproj
# !pip3 install gdal

print("\n\nAll required Dependancies are available.\n\n\n")

# %% [markdown]
# Able to open a file dialog box for the user to select the *Pre-Processed* flight data files that are to be queried for the ground elevation at the respective Latitudes and Longitudes.
# 
# This program only supports .csv files
# 
# Below is the expected .csv file format.

# %% [markdown]
# ![image.png](attachment:image.png)

# %% [markdown]
# **Note: Order of titles does not matter, only the order of the rows and the order of the first column.**
# 
# **Note: The program is case sensitive.**

# %%
# Importing Required Packages foruser to select .csv files

import tkinter
from tkinter import filedialog

# %%
# Function to open a file dialog box to select the .csv files

import tkinter
from tkinter import filedialog

def open_file_dialog():
    root = tkinter.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    file_path = filedialog.askopenfilenames(title="Select Files", filetypes=(("CSV files", "*.csv"), ("All files", "*.*")))
    return file_path

# %%
# Function to open a file dialog box to select the save location after files have procurred their ground elevation.

import tkinter
from tkinter import filedialog

def open_file_explorer():
    root = tkinter.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    directory_path = filedialog.askdirectory(title="Select Directory to Save Processed Files")
    return directory_path

# %%
# This function is to retrieve download URLs for the required DEM images. 
# Returns None if API fails to produce all the required links with 100 retry attempts

import requests
import time

def get_download_urls(max_latitude, max_longitude, latitude, longitude):
    url = "https://tnmaccess.nationalmap.gov/api/v1/products"
    params = {
        'datasets': 'National Elevation Dataset (NED) 1/3 arc-second',
        'bbox' : f'{longitude}, {latitude}, {max_longitude}, {max_latitude}',
        'prodFormats' : 'GeoTIFF',
        'dateType' : 'lastUpdated',
        'format': 'GeoTIFF',
    }
    retries = 100
    download_links = []
    while retries > 0:
        response = requests.get(url, params=params)
        if response.status_code == 200:
            try:
                data = response.json()
                for item in data.get('items', []):
                    if 'downloadURL' in item:
                        download_links.append(item['downloadURL'])
                    elif 'urls' in item and 'TIFF' in item['urls']:
                        download_links.append(item['urls']['TIFF'])
                break  # Exit loop if request is successful
            except requests.exceptions.JSONDecodeError:
                print(response.text)  # Print raw text for inspection
        else:
            retries -= 1
            time.sleep(0.5)  # Wait for 0.5 second before retrying
    if retries == 0:
        print("Maximum retries reached. API Failed.")
        return None
    else:
        return download_links

# %%
import requests
import time
import tqdm
import os
import pandas

def dem(max_latitude, max_longitude, min_latitude, min_longitude):
    print("Preparing to download DEM images...")
    current_dir = os.getcwd()
    dem_images_dir = os.path.join(current_dir, "DEM Images")
    if not os.path.exists(dem_images_dir):
        os.makedirs(dem_images_dir)
        print(f"Created directory: {dem_images_dir}")
    else:
        print(f"Directory already exists: {dem_images_dir}")

    print("Fetching download URLs for DEM images...")
    download_links = get_download_urls(max_latitude, max_longitude, min_latitude, min_longitude)
    if download_links == None:
        print("API Requests Failed! Please retry after a while.")
        return None
    else:
        # Get the Locations from the URLs
        locations = [None]*len(download_links)
    for i in range(len(download_links)):
        location = download_links[i]
        location = location[:-12]
        locations[i] = location[-8:-1]

    # Get the date from the URLs
    dates = [None]*len(download_links)
    for i in range(len(download_links)):
        date = download_links[i]
        dates[i] = date[-12:-4]

    latitudes = []
    longitudes = []
    for loc in locations:
        lat_dir = loc[0]  # First character indicates latitude direction ('n' or 's')
        lon_dir_index = loc.find('w') if 'w' in loc else loc.find('e')  # Find 'w' or 'e' for longitude direction
        lon_dir = loc[lon_dir_index]  # 'w' or 'e'

        # Extract latitude and longitude values
        latitude = int(loc[1:lon_dir_index])  # Latitude value is between the first character and the longitude direction character
        longitude = int(loc[lon_dir_index + 1:])  # Longitude value is after the longitude direction character

        # Adjust signs based on direction indicators
        if lat_dir == 's':
            latitude = -latitude
        if lon_dir == 'w':
            longitude = -longitude

        latitudes.append(latitude)
        longitudes.append(longitude)
    
    # Create a dictionary to store the latest entry for each identifier
    toDownload = pandas.DataFrame(
        {
            'URLs' : download_links,
            'Locations' : locations,
            'Latitudes' : latitudes,
            'Longitudes' : longitudes,
            'Dates' : dates
        }
    )

    toDownload['Dates'] = pandas.to_datetime(toDownload['Dates'], format = '%Y%m%d')
    toDownload.sort_values(by=['Locations', 'Dates'])
    toDownload = toDownload.drop_duplicates(subset='Locations', keep='last')
    toDownload.reset_index(drop=True, inplace=True)
    rows_to_drop = toDownload[(toDownload['Longitudes'] < min_longitude) | (toDownload['Longitudes'] > max_longitude)].index
    toDownload.drop(rows_to_drop, inplace=True)
    rows_to_drop = toDownload[(toDownload['Latitudes'] < min_latitude) | (toDownload['Latitudes'] > max_latitude)].index
    toDownload.drop(rows_to_drop, inplace=True)
    existing_files = os.listdir('DEM Images')
    existing_files = [filename[:-4] for filename in existing_files]
    toDownload = toDownload[~toDownload['Locations'].isin(existing_files)]
    toDownload.reset_index(drop=True, inplace=True)
    urls = toDownload['URLs'].tolist()
    filenames = toDownload['Locations'].tolist()

    num_retries = 100
    if len(urls) == 0:
        print("\nAll Required Files are Available.")
    else:
        for url, filename in zip(urls, filenames):
            retries = 0
            success = False
            while not success and retries < num_retries:
                response = requests.get(url, stream=True)
                if response.status_code == 200:
                    total_size = int(response.headers.get('content-length', 0))
                    file_path = os.path.join('DEM Images', filename + '.tif')
                    with open(file_path, 'wb') as file:
                        with tqdm.tqdm(total=total_size, unit='B', unit_scale=True, desc=f"Downloading {filename}.tif") as pbar:
                            for data in response.iter_content(chunk_size=1024):
                                file.write(data)
                                pbar.update(len(data))
                    success = True
                else:
                    print(f"Failed to download '{filename}.tif'. Status code: {response.status_code}. Retrying...")
                    retries += 1
                    time.sleep(0.25)  # Add a 0.25-second delay before retrying
            if not success:
                print(f"Failed to download '{filename}.tif' after {num_retries} retries.")
        print("Download completed!")

# %%
import os
import rasterio
import pandas
def save_bounds():
    global bounds_of_files
    existing_files = os.listdir('DEM Images')
    bounds_data = []
    for file_name in existing_files:
        if file_name.endswith('.tif'):
            file_path = os.path.join('DEM Images', file_name)
            with rasterio.open(file_path) as dataset:
                bounds = dataset.bounds
            bounds_data.append({
                'File Names': file_name,
                'Left Longitude': bounds.left,
                'Lower Latitude': bounds.bottom,
                'Right Longitude': bounds.right,
                'Upper Latitude': bounds.top
            })
    bounds_of_files = pandas.DataFrame(bounds_data)

# %%
def identify_sources(latitudes, longitudes):
    global bounds_of_files
    files = bounds_of_files["File Names"].tolist()
    left_longitude = bounds_of_files["Left Longitude"].tolist()
    lower_latitude = bounds_of_files["Lower Latitude"].tolist()
    right_longitude = bounds_of_files["Right Longitude"].tolist()
    upper_latitude = bounds_of_files["Upper Latitude"].tolist()
    references = [None] * len(latitudes)
    
    for i in range(len(references)):
        for j in range(len(files)):
            if (left_longitude[j] <= longitudes[i] <= right_longitude[j] and 
                lower_latitude[j] <= latitudes[i] <= upper_latitude[j]):
                references[i] = files[j]
                break
        if references[i] is None:
            references[i] = 'USE EPQS'
    return references


# %%
from osgeo import gdal
import os
import time
import requests
from osgeo import gdal, osr
import numpy as np
import pandas as pd
from pyproj import Proj, transform

def elevation(latitudes, longitudes):
    print("Identifying sources for the given coordinates...")
    dem_folder = 'DEM Images'
    references = identify_sources(latitudes, longitudes)
    elevations = [None]*len(references)
    i = 0
    while i < len(references):
        if elevations[i] == None:
            file_path = os.path.join(dem_folder, references[i])
            dataset = gdal.Open(file_path)
            dataset_proj = osr.SpatialReference()
            dataset_proj.ImportFromWkt(dataset.GetProjection())
            wgs84 = osr.SpatialReference()
            wgs84.ImportFromEPSG(4326)
            transform_proj = osr.CoordinateTransformation(wgs84, dataset_proj)
            
            j = i
            while j < len(references):
                if references[j] == references[i]:
                    x, y, _ = transform_proj.TransformPoint(longitudes[j], latitudes[j])
                    geo_transform = dataset.GetGeoTransform()
                    col = int((x - geo_transform[0]) / geo_transform[1])
                    row = int((y - geo_transform[3]) / geo_transform[5])
                    band = dataset.GetRasterBand(1)
                    elevations[j] = band.ReadAsArray(col, row, 1, 1)[0][0]
                j = j + 1
            dataset = None  # Close the dataset
        i += 1
        
    print("Fetching elevations from EPQS for missing data points...")
    i = 0
    while i < len(references):
        if references[i] == 'USE EPQS':
            url = "https://epqs.nationalmap.gov/v1/json?"
            params = {"x": longitudes[i], "y": latitudes[i], "units": "Meters"}
            retries = 100
            while retries > 0:
                try:
                    response = requests.get(url, params=params)
                    if response.status_code == 200:
                        vals = response.json()
                        elevations[i] = vals["value"]
                        break  # Exit retry loop on success
                except Exception as e:
                    print(f"An error occurred: {str(e)}")
                retries -= 1
                time.sleep(1)  # Wait for 1s
        i += 1
        
    return elevations

# %%
import pandas
import math
import numpy
bounds_of_files = pandas.DataFrame()
def main():
    print("Please select the CSV files to process.")
    files = open_file_dialog()
    print("Please select the directory to save the processed files.")
    save = open_file_explorer()
    for file in files:
        rows = []
        with open(file, 'r') as file:
            for line in file:
                fields = line.strip().split(',')
                rows.append(fields)
        max_fields = max(len(row) for row in rows)
        for row in rows:=
            while len(row) < max_fields:
                row.append(None)
        data = pandas.DataFrame(rows)
        data.columns = data.loc[3].tolist()
        latitude = data["latitude"].tolist()
        longitude = data["longitude"].tolist()
        time = data["time"].tolist()
        del time[0:9]
        del latitude[0:9]
        del longitude[0:9]
        latitude = numpy.asarray(latitude, dtype = float)
        longitude = numpy.asarray(longitude, dtype = float)
        print("Downloading Required Files")
        dem(math.ceil(max(latitude)), math.ceil(max(longitude)), math.floor(min(latitude)), math.floor(min((longitude))))
        save_bounds()
        print("Calculating elevations for the given coordinates...")
        elevations = elevation(latitude,longitude)
        elevations = numpy.array(elevations)
        elevations = elevations * 3.28084  # Convert to feet
        ground_elevation = data["ground_elevation"].tolist()
        del ground_elevation[9:]
        ground_elevation.extend(elevations)
        data["ground_elevation"] = ground_elevation
        file_path = file.name if hasattr(file, 'name') else str(file)
        data.to_csv(os.path.join(save, os.path.splitext(os.path.basename(file_path))[0] + "DEM.csv"), header = False, index= False)
        print(os.path.splitext(os.path.basename(file_path))[0] + " is complete.")
    print("\n\n All Files Complete!")

if __name__ == "__main__":
    main()

### **Digital Elevation Model Without Download**

In [ ]:
# %% [markdown]
# ### **United State Geological Survey Digital Elevation Model Query**

# %% [markdown]
# The following program uses Python to access USGS Digital Elevation Models (DEMs) to query elevation data for flight data.

# %% [markdown]
# Downloading Required Dependancies

# %%
!pip3 install tkinter
!pip3 install os
!pip3 install math
!pip3 install requests
!pip3 install time
!pip3 install pandas
!pip3 install tqdm
!pip3 install rasterio
!pip3 install pyproj
# !pip3 install gdal

print("\n\nAll required Dependancies are available.\n\n\n")

# %% [markdown]
# Able to open a file dialog box for the user to select the *Pre-Processed* flight data files that are to be queried for the ground elevation at the respective Latitudes and Longitudes.
# 
# This program only supports .csv files
# 
# Below is the expected .csv file format.

# %% [markdown]
# ![image.png](attachment:image.png)

# %% [markdown]
# **Note: Order of titles does not matter, only the order of the rows and the order of the first column.**
# 
# **Note: The program is case sensitive.**

# %%
# Importing Required Packages foruser to select .csv files

import tkinter
from tkinter import filedialog

# %%
# Function to open a file dialog box to select the .csv files

import tkinter
from tkinter import filedialog

def open_file_dialog():
    root = tkinter.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    file_path = filedialog.askopenfilenames(title="Select Files", filetypes=(("CSV files", "*.csv"), ("All files", "*.*")))
    return file_path

# %%
# Function to open a file dialog box to select the save location after files have procurred their ground elevation.

import tkinter
from tkinter import filedialog

def open_file_explorer():
    root = tkinter.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    directory_path = filedialog.askdirectory(title="Select Directory to Save Processed Files")
    return directory_path

# %%
# This function is to retrieve download URLs for the required DEM images. 
# Returns None if API fails to produce all the required links with 100 retry attempts

import requests
import time

def get_download_urls(max_latitude, max_longitude, latitude, longitude):
    url = "https://tnmaccess.nationalmap.gov/api/v1/products"
    params = {
        'datasets': 'National Elevation Dataset (NED) 1/3 arc-second',
        'bbox' : f'{longitude}, {latitude}, {max_longitude}, {max_latitude}',
        'prodFormats' : 'GeoTIFF',
        'dateType' : 'lastUpdated',
        'format': 'GeoTIFF',
    }
    retries = 100
    download_links = []
    while retries > 0:
        response = requests.get(url, params=params)
        if response.status_code == 200:
            try:
                data = response.json()
                for item in data.get('items', []):
                    if 'downloadURL' in item:
                        download_links.append(item['downloadURL'])
                    elif 'urls' in item and 'TIFF' in item['urls']:
                        download_links.append(item['urls']['TIFF'])
                break  # Exit loop if request is successful
            except requests.exceptions.JSONDecodeError:
                print(response.text)  # Print raw text for inspection
        else:
            retries -= 1
            time.sleep(0.5)  # Wait for 0.5 second before retrying
    if retries == 0:
        print("Maximum retries reached. API Failed.")
        return None
    else:
        return download_links

# %%
import requests
import time
import tqdm
import os
import pandas

def dem(max_latitude, max_longitude, min_latitude, min_longitude):
    print("Preparing to download DEM images...")
    current_dir = os.getcwd()
    dem_images_dir = os.path.join(current_dir, "DEM Images")
    if not os.path.exists(dem_images_dir):
        os.makedirs(dem_images_dir)
        print(f"Created directory: {dem_images_dir}")
    else:
        print(f"Directory already exists: {dem_images_dir}")

    print("Fetching download URLs for DEM images...")
    download_links = get_download_urls(max_latitude, max_longitude, min_latitude, min_longitude)
    if download_links == None:
        print("API Requests Failed! Please retry after a while.")
        return None
    else:
        # Get the Locations from the URLs
        locations = [None]*len(download_links)
    for i in range(len(download_links)):
        location = download_links[i]
        location = location[:-12]
        locations[i] = location[-8:-1]

    # Get the date from the URLs
    dates = [None]*len(download_links)
    for i in range(len(download_links)):
        date = download_links[i]
        dates[i] = date[-12:-4]

    latitudes = []
    longitudes = []
    for loc in locations:
        lat_dir = loc[0]  # First character indicates latitude direction ('n' or 's')
        lon_dir_index = loc.find('w') if 'w' in loc else loc.find('e')  # Find 'w' or 'e' for longitude direction
        lon_dir = loc[lon_dir_index]  # 'w' or 'e'

        # Extract latitude and longitude values
        latitude = int(loc[1:lon_dir_index])  # Latitude value is between the first character and the longitude direction character
        longitude = int(loc[lon_dir_index + 1:])  # Longitude value is after the longitude direction character

        # Adjust signs based on direction indicators
        if lat_dir == 's':
            latitude = -latitude
        if lon_dir == 'w':
            longitude = -longitude

        latitudes.append(latitude)
        longitudes.append(longitude)
    
    # Create a dictionary to store the latest entry for each identifier
    toDownload = pandas.DataFrame(
        {
            'URLs' : download_links,
            'Locations' : locations,
            'Latitudes' : latitudes,
            'Longitudes' : longitudes,
            'Dates' : dates
        }
    )

    toDownload['Dates'] = pandas.to_datetime(toDownload['Dates'], format = '%Y%m%d')
    toDownload.sort_values(by=['Locations', 'Dates'])
    toDownload = toDownload.drop_duplicates(subset='Locations', keep='last')
    toDownload.reset_index(drop=True, inplace=True)
    rows_to_drop = toDownload[(toDownload['Longitudes'] < min_longitude) | (toDownload['Longitudes'] > max_longitude)].index
    toDownload.drop(rows_to_drop, inplace=True)
    rows_to_drop = toDownload[(toDownload['Latitudes'] < min_latitude) | (toDownload['Latitudes'] > max_latitude)].index
    toDownload.drop(rows_to_drop, inplace=True)
    existing_files = os.listdir('DEM Images')
    existing_files = [filename[:-4] for filename in existing_files]
    toDownload = toDownload[~toDownload['Locations'].isin(existing_files)]
    toDownload.reset_index(drop=True, inplace=True)
    urls = toDownload['URLs'].tolist()
    filenames = toDownload['Locations'].tolist()

    num_retries = 100
    if len(urls) == 0:
        print("\nAll Required Files are Available.")
    else:
        for url, filename in zip(urls, filenames):
            retries = 0
            success = False
            while not success and retries < num_retries:
                response = requests.get(url, stream=True)
                if response.status_code == 200:
                    total_size = int(response.headers.get('content-length', 0))
                    file_path = os.path.join('DEM Images', filename + '.tif')
                    with open(file_path, 'wb') as file:
                        with tqdm.tqdm(total=total_size, unit='B', unit_scale=True, desc=f"Downloading {filename}.tif") as pbar:
                            for data in response.iter_content(chunk_size=1024):
                                file.write(data)
                                pbar.update(len(data))
                    success = True
                else:
                    print(f"Failed to download '{filename}.tif'. Status code: {response.status_code}. Retrying...")
                    retries += 1
                    time.sleep(0.25)  # Add a 0.25-second delay before retrying
            if not success:
                print(f"Failed to download '{filename}.tif' after {num_retries} retries.")
        print("Download completed!")

# %%
import os
import rasterio
import pandas
def save_bounds():
    global bounds_of_files
    existing_files = os.listdir('DEM Images')
    bounds_data = []
    for file_name in existing_files:
        if file_name.endswith('.tif'):
            file_path = os.path.join('DEM Images', file_name)
            with rasterio.open(file_path) as dataset:
                bounds = dataset.bounds
            bounds_data.append({
                'File Names': file_name,
                'Left Longitude': bounds.left,
                'Lower Latitude': bounds.bottom,
                'Right Longitude': bounds.right,
                'Upper Latitude': bounds.top
            })
    bounds_of_files = pandas.DataFrame(bounds_data)

# %%
def identify_sources(latitudes, longitudes):
    global bounds_of_files
    files = bounds_of_files["File Names"].tolist()
    left_longitude = bounds_of_files["Left Longitude"].tolist()
    lower_latitude = bounds_of_files["Lower Latitude"].tolist()
    right_longitude = bounds_of_files["Right Longitude"].tolist()
    upper_latitude = bounds_of_files["Upper Latitude"].tolist()
    references = [None] * len(latitudes)
    
    for i in range(len(references)):
        for j in range(len(files)):
            if (left_longitude[j] <= longitudes[i] <= right_longitude[j] and 
                lower_latitude[j] <= latitudes[i] <= upper_latitude[j]):
                references[i] = files[j]
                break
        if references[i] is None:
            references[i] = 'USE EPQS'
    return references


# %%
from osgeo import gdal
import os
import time
import requests
from osgeo import gdal, osr
import numpy as np
import pandas as pd
from pyproj import Proj, transform

def elevation(latitudes, longitudes):
    print("Identifying sources for the given coordinates...")
    dem_folder = 'DEM Images'
    references = identify_sources(latitudes, longitudes)
    elevations = [None]*len(references)
    i = 0
    while i < len(references):
        if elevations[i] == None:
            file_path = os.path.join(dem_folder, references[i])
            dataset = gdal.Open(file_path)
            dataset_proj = osr.SpatialReference()
            dataset_proj.ImportFromWkt(dataset.GetProjection())
            wgs84 = osr.SpatialReference()
            wgs84.ImportFromEPSG(4326)
            transform_proj = osr.CoordinateTransformation(wgs84, dataset_proj)
            
            j = i
            while j < len(references):
                if references[j] == references[i]:
                    x, y, _ = transform_proj.TransformPoint(longitudes[j], latitudes[j])
                    geo_transform = dataset.GetGeoTransform()
                    col = int((x - geo_transform[0]) / geo_transform[1])
                    row = int((y - geo_transform[3]) / geo_transform[5])
                    band = dataset.GetRasterBand(1)
                    elevations[j] = band.ReadAsArray(col, row, 1, 1)[0][0]
                j = j + 1
            dataset = None  # Close the dataset
        i += 1
        
    print("Fetching elevations from EPQS for missing data points...")
    i = 0
    while i < len(references):
        if references[i] == 'USE EPQS':
            url = "https://epqs.nationalmap.gov/v1/json?"
            params = {"x": longitudes[i], "y": latitudes[i], "units": "Meters"}
            retries = 100
            while retries > 0:
                try:
                    response = requests.get(url, params=params)
                    if response.status_code == 200:
                        vals = response.json()
                        elevations[i] = vals["value"]
                        break  # Exit retry loop on success
                except Exception as e:
                    print(f"An error occurred: {str(e)}")
                retries -= 1
                time.sleep(1)  # Wait for 1s
        i += 1
        
    return elevations

# %%
import pandas
import math
import numpy
bounds_of_files = pandas.DataFrame()
def main():
    print("Please select the CSV files to process.")
    files = open_file_dialog()
    print("Please select the directory to save the processed files.")
    save = open_file_explorer()
    for file in files:
        rows = []
        with open(file, 'r') as file:
            for line in file:
                fields = line.strip().split(',')
                rows.append(fields)
        max_fields = max(len(row) for row in rows)
        for row in rows:
            while len(row) < max_fields:
                row.append(None)
        data = pandas.DataFrame(rows)
        data.columns = data.loc[3].tolist()
        latitude = data["latitude"].tolist()
        longitude = data["longitude"].tolist()
        time = data["time"].tolist()
        del time[0:9]
        del latitude[0:9]
        del longitude[0:9]
        latitude = numpy.asarray(latitude, dtype = float)
        longitude = numpy.asarray(longitude, dtype = float)
        print("Downloading Required Files")
        dem(math.ceil(max(latitude)), math.ceil(max(longitude)), math.floor(min(latitude)), math.floor(min((longitude))))
        save_bounds()
        print("Calculating elevations for the given coordinates...")
        elevations = elevation(latitude,longitude)
        elevations = numpy.array(elevations)
        elevations = elevations * 3.28084  # Convert to feet
        ground_elevation = data["ground_elevation"].tolist()
        del ground_elevation[9:]
        ground_elevation.extend(elevations)
        data["ground_elevation"] = ground_elevation
        file_path = file.name if hasattr(file, 'name') else str(file)
        data.to_csv(os.path.join(save, os.path.splitext(os.path.basename(file_path))[0] + "DEM.csv"), header = False, index= False)
        print(os.path.splitext(os.path.basename(file_path))[0] + " is complete.")
    print("\n\n All Files Complete!")

if __name__ == "__main__":
    main()